## Trim `karma_edges` to v3

`karma_edges.csv` is a directed edge list `source_id → target_id → karma_score` produced by the LLM run in `llm.ipynb`. The candidate edges came from v1's `affiliated` column, which included links inside the `Family` section — multi-century genealogy that doesn't represent storyline interaction.

`scrape_characters_v3.ipynb` rebuilt `affiliated` with the Family section excluded (→ `characters_enriched_v3.csv`). This notebook produces `karma_edges_v3.csv` by keeping only the karma rows whose directed pair `(source_id, target_id)` survived in v3: i.e. `target_id` still appears in `source_id`'s v3 affiliated list.

Direction matters. If A→B disappeared in v3 (B was in A's Family section) but B→A is still present (B's article references A in History/Recent Events), we keep only B→A. Symmetric karma is not assumed.

In [1]:
import pandas as pd

karma = pd.read_csv('../csvs/karma_edges.csv')
v3 = pd.read_csv('../csvs/characters_enriched_v3.csv').fillna('')

# Build the set of directed (source, target) pairs that survive in v3.
v3_edges = set()
for _, row in v3.iterrows():
    src = row['ID']
    aff = row['affiliated']
    if not aff:
        continue
    for tgt in aff.split(';'):
        tgt = tgt.strip()
        if tgt and tgt != src:
            v3_edges.add((src, tgt))

pairs = list(zip(karma['source_id'], karma['target_id']))
mask = [pair in v3_edges for pair in pairs]
karma_v3 = karma[mask].copy()
karma_v3.to_csv('../csvs/karma_edges_v3.csv', index=False)

print(f'v1 karma edges:           {len(karma):,}')
print(f'v3 karma edges:           {len(karma_v3):,}')
print(f'Dropped (Family-section):  {len(karma) - len(karma_v3):,}  ({(1 - len(karma_v3)/len(karma)) * 100:.1f}%)')
print(f'\nDirected (src,tgt) pairs in v3 graph total: {len(v3_edges):,}')


v1 karma edges:           10,735
v3 karma edges:           3,966
Dropped (Family-section):  6,769  (63.1%)

Directed (src,tgt) pairs in v3 graph total: 23,322


### Sanity: where did the drops come from?
Show the characters whose outgoing karma edges shrank the most. These should be the dynastic figures whose v1 article had a heavy `Family` section (Targaryens, Starks, Lannisters, Freys).

In [2]:
before = karma.groupby('source_id').size()
after = karma_v3.groupby('source_id').size()
delta = (before - after.reindex(before.index, fill_value=0)).sort_values(ascending=False)

print('Top 15 sources by edges dropped:')
print(delta.head(15).to_string())

# karma_edges.csv has ~31 malformed karma_score values (empty, 'w', '5w', a CSV-parse
# artifact like '_the_Hawk') and uses a 0-10 scale, not 0-100. Coerce to numeric and
# drop invalid rows before bucketing.
def bucket(s):
    if s <= 3:
        return 'enemy'
    if s >= 7:
        return 'friend'
    return 'neutral'


def score_buckets(df):
    s = pd.to_numeric(df['karma_score'], errors='coerce').dropna()
    return s.apply(bucket).value_counts().to_dict()


print('\nKarma-score distribution before/after (scale 0-10, enemy <=3, friend >=7):')
print('v1:', score_buckets(karma))
print('v3:', score_buckets(karma_v3))

n_bad_v1 = pd.to_numeric(karma['karma_score'], errors='coerce').isna().sum()
n_bad_v3 = pd.to_numeric(karma_v3['karma_score'], errors='coerce').isna().sum()
print(f'\nNon-numeric karma_score rows skipped:  v1 = {n_bad_v1}, v3 = {n_bad_v3}')


Top 15 sources by edges dropped:
source_id
Robert_Flowers           143
Demon_of_Darry           143
Malleon                  139
Greydon                  139
Lucan_(Grand_Maester)    139
Clegg                    137
Walder_Frey              136
Aethelmure               135
Viserys_II_Targaryen     123
Mace_Tyrell              112
Barristan_Selmy          111
Tyrion_Lannister         110
Eddard_Stark             107
Corlys_Velaryon          103
Cregan_Stark             102

Karma-score distribution before/after (scale 0-10, enemy <=3, friend >=7):
v1: {'neutral': 6158, 'enemy': 2953, 'friend': 1593}
v3: {'neutral': 1851, 'enemy': 1078, 'friend': 1023}

Non-numeric karma_score rows skipped:  v1 = 31, v3 = 14
